In [1]:
import os

directory_path = "torch_helper_functions"
subdirectory_name = "multilabel"
full_path = os.path.join(directory_path, subdirectory_name)

if os.path.exists(full_path):
    print("Directory Already Exists")
else:
    os.makedirs(full_path)
    print(f"Created `{full_path}` directory")


Created `torch_helper_functions/multilabel` directory


In [2]:
# !pip install torchmetrics
# !pip install wandb onnx -Uq

In [3]:
%%writefile torch_helper_functions/multilabel/data_cleaner.py
"""
Contains functionality for cleaning data related to data kept in folders and it's labels are in dataframe.

## Functions:
  `preprocess_image_paths`:Preprocesses image paths in a DataFrame by replacing parts of the paths and changing extensions.
  `valid_image_paths`:Preprocesses image paths in a DataFrame by removing all the invalid non-existing images present in the original DataFrame.
"""
import os
import pandas as pd
import numpy as np
def preprocess_image_paths(df:pd.DataFrame,
                           image_path_column: str,
                           orignal_path: str,
                           new_path: str,
                           old_extension: str,
                           new_extension: str = None,
                           new_image_path_column: str = None)->pd.DataFrame:
  """
  #### Preprocesses image paths in a DataFrame by replacing parts of the paths and changing extensions.

  ## Parameters:
    `df (pd.DataFrame)`: The DataFrame containing image paths.
    `image_path_column (str)`: The name of the column in df containing the image paths.
    `new_image_path_column (str)`: The name of the column in df where to save the new image paths
                                           if none it will just replce orignal path.
    `original_path (str)`: The part of the image path to be replaced in dataframe diffrent from your current data location.
    `new_path (str)`: The replacement for the original_path in dataframe for new data location.
    `old_extension (str)`: The extension of the image paths to be replaced.
    `new_extension (str)`: The new extension to replace the old extension if none keeps same extension.
  ## Returns:
    pd.DataFrame: The function returns modified DataFrame.
  ## Example:
      preprocess_image_paths(df,"image_paths","D://kaggle//","D://root","jpg","jpeg","new_image_paths")
  """
  if new_image_path_column is None:
    new_image_path_column = image_path_column
  if new_extension is None:
    new_extension = old_extension
  df[new_image_path_column] = df[image_path_column].str.replace(orignal_path, new_path)
  df[new_image_path_column] = df[image_path_column].str.replace(old_extension, new_extension)
  return df

def validate_image_paths(df: pd.DataFrame,
                         image_path_column: str,
                         folder_path: str,
                         save_path: str="root")->pd.DataFrame:
  """
  #### Preprocesses image paths in a DataFrame by removing all the invalid non-existing images present in the original DataFrame.

  ## Parameters:
    `df (pd.DataFrame)`: The DataFrame containing image paths.
    `image_path_column (str)`: The name of the column in df containing the image paths.
    `folder_path (str)`: The path to the folder containing the images.
    `save_path (str)=default="root"`: The path to save a numpy file contatining list of valid images if not path just filename.

  ## Returns:
    pd.DataFrame: The function returns modified dataframe and also save a numpy file containing valid images names.
  ## Example:
     validate_image_paths(df,"image_paths","D://train_image_data","D://valid_image//valid_image_paths")
  """
  if save_path == "root":
    save_path = os.path.join(os.getcwd(), "valid_image_paths.npy")
  valid_images = []
  for index, row in df.iterrows():
    image_path = os.path.join(folder_path, row[image_path_column])
    # Check if the file exists
    if os.path.isfile(image_path):
      valid_images.append(row[image_path_column])
  valid_images = np.array(valid_images)
  np.save(save_path,valid_images)
  # Filter the DataFrame to include only rows with valid image paths
  df = df[df[image_path_column].isin(valid_images)]
  return df

Writing torch_helper_functions/multilabel/data_cleaner.py


In [4]:
%%writefile torch_helper_functions/utils.py
"""
Files contains utility functions for PyTorch model training.
"""
from pathlib import Path
import torch
from typing import Dict,List
import pandas as pd
import torch
from torch import nn,sigmoid,softmax,cuda,inference_mode
from torch.utils.data import DataLoader
from torchmetrics.classification import BinaryAccuracy,MulticlassAccuracy,BinaryF1Score,MulticlassF1Score
from tabulate import tabulate
from tqdm.auto import tqdm

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
  """
  #### Saves a PyTorch model to a target directory.

  ## Parameters:
    `model`: A target PyTorch model to save.
    `target_dir`: A directory for saving the model to.
    `model_name`: A filename for the saved model. Should include
      either ".pth" or ".pt" as the file extension.

  ## Return:
    None: This only saves the state dict(model weights) of
          the model that can be then used load in same model architecture.
  ## Example:
    save_model(model=model_0,
               target_dir="models",
               model_name="05_going_modular_tingvgg_model.pth")
  """
  # Create target directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)

  # Create model save path
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  # Save the model state_dict()
  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)

def calculate_class_weights(dataframe:pd.DataFrame,
                            label_columns:List[str]):
    """
    #### Calculate normalized class weights based on the frequencies of each class in the label columns of the dataframe.

    ## Args:
      `dataframe (pd.DataFrame)`: The dataframe containing the data.
      `label_columns (list)`: A list of column names containing labels.

    ## Returns:
     `dict`: A dictionary containing the normalized class weights.
     `list`: A list containing weights of classes for loss function.
    ## Example:
    Suppose we have a dataframe 'df' with the following structure:
    ```
          label_column
    0      class1
    1      class2
    2      class1
    3      class3
    4      class2
    ```
    We can calculate the class weights using:
    ```
    class_weights_dict,class_weights = calculate_class_weights(df, ['label_column'])
    ```
    This will return a dictionary like:
    ```
    {'class1': 1.0, 'class2': 2.0, 'class3': 2.0}
    ```
    Also class weight tensor like list for loss_fn.
    [1.0,2.0,2.0]
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    class_frequencies = {}

    # Calculate class frequencies
    for label in label_columns:
        class_frequencies[label] = dataframe[label].sum()

    # Calculate total samples
    total_samples = sum(class_frequencies.values())

    # Calculate class weights
    class_weights = {cls: total_samples / freq for cls, freq in class_frequencies.items()}

    # Normalize class weights
    max_weight = max(class_weights.values())
    class_weights_normalized = {cls: max_weight / freq for cls, freq in class_frequencies.items()}

    # Convert to PyTorch tensor
    weights_tensor = torch.tensor(list(class_weights_normalized.values()),device=device).float()
    return class_weights_normalized, weights_tensor

from tabulate import tabulate
# Define a function to print the DataFrame after each epoch
def print_model_train_results(results):
    df_results = pd.DataFrame(results)
    # Set 'Type' as the index
    df_results.set_index("Results", inplace=True)
    print(tabulate(df_results, headers='keys', tablefmt='fancy_grid'),"\n")

def model_evaluation(model: nn.Module,
                    dataloader: DataLoader,
                    criterion_bowel: nn.Module,
                    criterion_extra: nn.Module,
                    criterion_kidney: nn.Module,
                    criterion_liver: nn.Module,
                    criterion_spleen: nn.Module):
    """
    Perform a model evaluation on testing data.

    Parameters:
        model (nn.Module): The neural network model.
        dataloader (torch.utils.data.DataLoader): DataLoader for the testing dataset.
        criterion_bowel (nn.Module): Loss function for bowel task.
        criterion_extra (nn.Module): Loss function for extra task.
        criterion_liver (nn.Module): Loss function for liver task.
        criterion_kidney (nn.Module): Loss function for kidney task.
        criterion_spleen (nn.Module): Loss function for spleen task.

    Returns:
        No print model evaluation on testing data
    """
    device = "cuda" if cuda.is_available() else "cpu"

    # Getting accuracy and f1score
    binary_acc = BinaryAccuracy().to(device)
    multi_acc = MulticlassAccuracy(num_classes=3).to(device)
    binary_f1 = BinaryF1Score().to(device)
    multi_f1 = MulticlassF1Score(num_classes=3).to(device)

    # Create testing metrics variables
    test_loss, test_acc_bowel, test_acc_extra, test_acc_kidney, test_acc_liver, test_acc_spleen = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    test_f1_bowel, test_f1_extra, test_f1_liver, test_f1_kidney, test_f1_spleen = 0.0, 0.0, 0.0, 0.0, 0.0

    # Set model to evaluation mode
    model.eval()

    # Iterate over testing dataloader
    with inference_mode():
        for batch_idx, (X, y_bowel, y_extra, y_kidney, y_liver, y_spleen) in tqdm(enumerate(dataloader), desc="Testing", total=len(dataloader)):
            # Move data to appropriate device
            X, y_bowel, y_extra, y_kidney,y_liver, y_spleen = X.to(device), y_bowel.to(device), y_extra.to(device), y_kidney.to(device), y_liver.to(device), y_spleen.to(device)

            # Forward pass
            bowel_out, extra_out, kidney_out,liver_out, spleen_out = model(X)

            y_bowel = y_bowel.view(-1,1)
            y_extra = y_extra.view(-1,1)
            # Calculate loss
            loss_bowel = criterion_bowel(bowel_out, y_bowel)
            loss_extra = criterion_extra(extra_out, y_extra)
            loss_kidney = criterion_kidney(kidney_out, y_kidney)
            loss_liver = criterion_liver(liver_out, y_liver)
            loss_spleen = criterion_spleen(spleen_out, y_spleen)

            # Accumulate loss
            test_loss += (loss_bowel.item() + loss_extra.item() + loss_kidney.item() + loss_liver.item() + loss_spleen.item())
            y_kidney=y_kidney.argmax(dim=1)
            y_liver=y_liver.argmax(dim=1)
            y_spleen=y_spleen.argmax(dim=1)
            # Compute accuracy for each task
            test_acc_bowel += binary_acc(sigmoid(bowel_out), y_bowel).item()
            test_acc_extra += binary_acc(sigmoid(extra_out), y_extra).item()
            test_acc_kidney += multi_acc(softmax(kidney_out,dim=1).argmax(dim=1), y_kidney).item()
            test_acc_liver += multi_acc(softmax(liver_out,dim=1).argmax(dim=1), y_liver).item()
            test_acc_spleen += multi_acc(softmax(spleen_out,dim=1).argmax(dim=1), y_spleen).item()

            # Compute F1 score for each task
            test_f1_bowel += binary_f1(sigmoid(bowel_out), y_bowel).item()
            test_f1_extra += binary_f1(sigmoid(extra_out), y_extra).item()
            test_f1_kidney += multi_f1(softmax(kidney_out,dim=1).argmax(dim=1), y_kidney).item()
            test_f1_liver += multi_f1(softmax(liver_out,dim=1).argmax(dim=1), y_liver).item()
            test_f1_spleen += multi_f1(softmax(spleen_out,dim=1).argmax(dim=1), y_spleen).item()

    # Compute average loss, accuracy, and F1 score
    num_batches = len(dataloader)
    test_loss /= (num_batches)
    test_acc_bowel /= num_batches
    test_acc_extra /= num_batches
    test_acc_kidney /= num_batches
    test_acc_liver /= num_batches
    test_acc_spleen /= num_batches
    test_f1_bowel /= num_batches
    test_f1_extra /= num_batches
    test_f1_kidney /= num_batches
    test_f1_liver /= num_batches
    test_f1_spleen /= num_batches

    print(f"Test loss is: {test_loss:.4f}")
    print_model_train_results({
    "Results":["Accuracy","F1_Score"],
    "bowel": [test_acc_bowel,test_f1_bowel],
    "extravation": [test_acc_extra,test_f1_extra],
    "kidney": [test_acc_kidney,test_f1_kidney],
    "liver": [test_acc_liver,test_f1_liver],
    "spleen": [test_acc_spleen,test_f1_spleen]
    })

Writing torch_helper_functions/utils.py


In [5]:
%%writefile torch_helper_functions/multilabel/train_engine.py
"""
Helper class for training and validation steps in a convolutional neural network for mutilabel classification
Also it will save all model progress in weight and bias platform.

## Attributes:
  None
## Methods:
  `train_step`: Performs a single training step.
  `validation_step`: Performs a single validation step.
  `train`: Trains the model for a specified number of epochs.
"""
from torch import cuda,onnx,sigmoid,softmax,round,inference_mode,randperm
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from typing import Tuple,Dict,List
from collections import defaultdict
import wandb
import torch.nn as nn
import torch.optim as optim
from torchmetrics.classification import BinaryAccuracy,MulticlassAccuracy
from torch_helper_functions import utils
import pandas as pd
import os
import numpy as np

def shuffle_minibatch(X, y_bowel, y_extra, y_kidney, y_liver, y_spleen):
    """
    Shuffle the minibatch of inputs and labels.

    Args:
        X (torch.Tensor): Input images.
        y_bowel (torch.Tensor): Labels for bowel task.
        y_extra (torch.Tensor): Labels for extra task.
        y_kidney (torch.Tensor): Labels for kidney task.
        y_liver (torch.Tensor): Labels for liver task.
        y_spleen (torch.Tensor): Labels for spleen task.

    Returns:
        tuple: Tuple containing shuffled input images and corresponding shuffled labels.
    """
    batch_size = X.size(0)
    idx = randperm(batch_size)

    X_shuffled = X[idx]
    y_bowel_shuffled = y_bowel[idx]
    y_extra_shuffled = y_extra[idx]
    y_kidney_shuffled = y_kidney[idx]
    y_liver_shuffled = y_liver[idx]
    y_spleen_shuffled = y_spleen[idx]

    return X_shuffled, y_bowel_shuffled, y_extra_shuffled, y_kidney_shuffled, y_liver_shuffled, y_spleen_shuffled

def train_step_cutmix(model: nn.Module,
               dataloader: DataLoader,
               criterion_bowel: nn.Module,
               criterion_extra: nn.Module,
               criterion_kidney: nn.Module,
               criterion_liver: nn.Module,
               criterion_spleen: nn.Module,
               binary_acc
               ,multi_acc
               ,optimizer: optim.Optimizer,
               CUTMIX_ALPHA):
    """ Perform a single training step with CutMix data augmentation.
    Parameters:
    model (nn.Module): The neural network model.
    dataloader (torch.utils.data.DataLoader): DataLoader for the training dataset.
    criterion_bowel (nn.Module): Loss function for bowel task.
    criterion_extra (nn.Module): Loss function for extra task.
    criterion_liver (nn.Module): Loss function for liver task.
    criterion_kidney (nn.Module): Loss function for kidney task.
    criterion_spleen (nn.Module): Loss function for spleen task.
    binary_acc: Function to compute binary accuracy.
    multi_acc: Function to compute multi-class accuracy.
    optimizer (optim.Optimizer): Optimizer for gradient descent.
    EPOCHS (int): Total number of training epochs.
    CUTMIX_ALPHA (float): Alpha parameter for CutMix.
    device (str): Device to use for training (e.g., 'cpu' or 'cuda').
    Returns:
    Tuple: Tuple containing training loss, accuracy for bowel, extra, liver, kidney, spleen, f1 score for bowel, extra, liver, kidney, spleen.
    """
    # Setting up device
    device = "cuda" if cuda.is_available() else "cpu"
    # Create train metrics variables
    train_loss, train_acc_bowel, train_acc_extra, train_acc_liver, train_acc_kidney, train_acc_spleen = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    # Creating training loop
    model.train()
    total_steps = len(dataloader)
    for i, (X, y_bowel, y_extra, y_kidney, y_liver, y_spleen) in tqdm(enumerate(dataloader), desc="Training", leave=False, total=len(dataloader)):
      X, y_bowel, y_extra, y_kidney, y_liver, y_spleen = X.to(device), y_bowel.to(device), y_extra.to(device), y_kidney.to(device), y_liver.to(device), y_spleen.to(device)
      _, C, H, W = X.size()
      # CutMix data augmentation
      cutmix_decision = np.random.rand()
      if cutmix_decision > 0.50:
        # Cutmix: https://arxiv.org/pdf/1905.04899.pdf
        X_shuffled, y_bowel_shuffled, y_extra_shuffled, y_kidney_shuffled, y_liver_shuffled, y_spleen_shuffled = shuffle_minibatch(X, y_bowel, y_extra, y_kidney, y_liver, y_spleen)
        lam = np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA)
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)

        # uniform
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        X[:, :, bbx1:bbx2, bby1:bby2] = X_shuffled[:, :, bbx1:bbx2, bby1:bby2]
        lam = 1 - (bbx2 - bbx1) * (bby2 - bby1) / (W * H)

      # 1. Forward pass
      bowel_out, extra_out, kidney_out, liver_out, spleen_out = model(X)

      # 2. Calculate loss/acc
      if cutmix_decision > 0.50:
        loss_bowel = criterion_bowel(bowel_out, y_bowel) * lam + criterion_bowel(bowel_out, y_bowel_shuffled) * (1. - lam)
        loss_extra = criterion_extra(extra_out, y_extra) * lam + criterion_extra(extra_out, y_extra_shuffled) * (1. - lam)
        loss_kidney = criterion_kidney(kidney_out, y_kidney) * lam + criterion_kidney(kidney_out, y_kidney_shuffled) * (1. - lam)
        loss_liver = criterion_liver(liver_out, y_liver) * lam + criterion_liver(liver_out, y_liver_shuffled) * (1. - lam)
        loss_spleen = criterion_spleen(spleen_out, y_spleen) * lam + criterion_spleen(spleen_out, y_spleen_shuffled) * (1. - lam)
      else:
        loss_bowel = criterion_bowel(bowel_out, y_bowel)
        loss_extra = criterion_extra(extra_out, y_extra)
        loss_kidney = criterion_kidney(kidney_out, y_kidney)
        loss_liver = criterion_liver(liver_out, y_liver)
        loss_spleen = criterion_spleen(spleen_out, y_spleen)

      loss = loss_bowel + loss_extra + loss_kidney + loss_liver + loss_spleen
      train_loss += loss.item()

      # Converting binary labels to true
      bowel_out = round(sigmoid(bowel_out))
      extra_out = round(sigmoid(extra_out))
      kidney_out = softmax(kidney_out, dim=1).argmax(dim=1)
      liver_out = softmax(liver_out, dim=1).argmax(dim=1)
      spleen_out = softmax(spleen_out, dim=1).argmax(dim=1)

      # Getting true labels for multiclass classification
      y_kidney = y_kidney.argmax(dim=1)
      y_liver = y_liver.argmax(dim=1)
      y_spleen = y_spleen.argmax(dim=1)

      # Compute accuracy for each task
      train_acc_bowel += binary_acc(bowel_out, y_bowel).item()
      train_acc_extra += binary_acc(extra_out, y_extra).item()
      train_acc_kidney += multi_acc(kidney_out, y_kidney).item()
      train_acc_liver += multi_acc(liver_out, y_liver).item()
      train_acc_spleen += multi_acc(spleen_out, y_spleen).item()

      # 3. Zero grad
      optimizer.zero_grad()

      # 4. Backward pass
      loss.backward()

      # 5. Optimizer step
      optimizer.step()
    num_batches = len(dataloader)
    train_loss /= num_batches
    train_acc_bowel /= num_batches
    train_acc_extra /= num_batches
    train_acc_kidney /= num_batches
    train_acc_liver /= num_batches
    train_acc_spleen /= num_batches

    return train_loss, train_acc_bowel, train_acc_extra, train_acc_kidney, train_acc_liver, train_acc_spleen

def train_step(model: nn.Module,
               dataloader: DataLoader,
               criterion_bowel: nn.Module,
               criterion_extra: nn.Module,
               criterion_kidney: nn.Module,
               criterion_liver: nn.Module,
               criterion_spleen: nn.Module,
               binary_acc,
               multi_acc,
               optimizer: optim.Optimizer):
    """
    Perform a single training step.

    Parameters:
        model (nn.Module): The neural network model.
        dataloader (torch.utils.data.DataLoader): DataLoader for the training dataset.
        criterion_bowel (nn.Module): Loss function for bowel task.
        criterion_extra (nn.Module): Loss function for extra task.
        criterion_liver (nn.Module): Loss function for liver task.
        criterion_kidney (nn.Module): Loss function for kidney task.
        criterion_spleen (nn.Module): Loss function for spleen task.
        optimizer (optim.Optimizer): Optimizer for gradient descent.

    Returns:
        Tuple: Tuple containing training loss, accuracy for bowel, extra, liver, kidney, spleen,
               f1 score for bowel, extra, liver, kidney, spleen.
    """
    device = "cuda" if cuda.is_available() else "cpu"
    # Create train metrics variables
    train_loss, train_acc_bowel, train_acc_extra, train_acc_liver, train_acc_kidney, train_acc_spleen = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    # Creating training loop
    model.train()
    for batch, (X, y_bowel, y_extra, y_kidney, y_liver, y_spleen) in tqdm(enumerate(dataloader), desc="Training", leave=False, total=len(dataloader)):
        X, y_bowel, y_extra, y_kidney, y_liver, y_spleen = X.to(device), y_bowel.to(device), y_extra.to(device), y_kidney.to(device), y_liver.to(device), y_spleen.to(device)
        # 1. Forward pass
        bowel_out, extra_out, kidney_out, liver_out, spleen_out = model(X)

        # 2. Calculate loss/acc
        loss_bowel = criterion_bowel(bowel_out, y_bowel)
        loss_extra = criterion_extra(extra_out, y_extra)
        loss_kidney = criterion_kidney(kidney_out, y_kidney)
        loss_liver = criterion_liver(liver_out, y_liver)
        loss_spleen = criterion_spleen(spleen_out, y_spleen)
        loss = loss_bowel + loss_extra + loss_kidney + loss_liver + loss_spleen
        train_loss += loss.item()

        # Converting binary labels to true
        bowel_out=round(sigmoid(bowel_out))
        extra_out=round(sigmoid(extra_out))
        kidney_out=softmax(kidney_out,dim=1).argmax(dim=1)
        liver_out=softmax(liver_out,dim=1).argmax(dim=1)
        spleen_out=softmax(spleen_out,dim=1).argmax(dim=1)
        # Getting true labels for muticlass classification
        y_kidney=y_kidney.argmax(dim=1)
        y_liver=y_liver.argmax(dim=1)
        y_spleen=y_spleen.argmax(dim=1)

        # Compute accuracy for each task
        train_acc_bowel += binary_acc(bowel_out, y_bowel).item()
        train_acc_extra += binary_acc(extra_out, y_extra).item()
        train_acc_kidney += multi_acc(kidney_out, y_kidney).item()
        train_acc_liver += multi_acc(liver_out, y_liver).item()
        train_acc_spleen += multi_acc(spleen_out, y_spleen).item()

        # 3. Zero grad
        optimizer.zero_grad()
        # 4. Backward pass
        loss.backward()
        # 5. Optimizer step
        optimizer.step()

    # Compute the average metrics
    num_batches = len(dataloader)
    train_loss /= num_batches
    train_acc_bowel /= num_batches
    train_acc_extra /= num_batches
    train_acc_kidney /= num_batches
    train_acc_liver /= num_batches
    train_acc_spleen /= num_batches

    return train_loss, train_acc_bowel, train_acc_extra, train_acc_kidney, train_acc_liver, train_acc_spleen


def validation_step(model: nn.Module,
                    dataloader: DataLoader,
                    criterion_bowel: nn.Module,
                    criterion_extra: nn.Module,
                    criterion_kidney: nn.Module,
                    criterion_liver: nn.Module,
                    criterion_spleen: nn.Module,
                    binary_acc,
                    multi_acc):
    """
    Perform a single validation step.

    Parameters:
        model (nn.Module): The neural network model.
        dataloader (torch.utils.data.DataLoader): DataLoader for the validation dataset.
        criterion_bowel (nn.Module): Loss function for bowel task.
        criterion_extra (nn.Module): Loss function for extra task.
        criterion_liver (nn.Module): Loss function for liver task.
        criterion_kidney (nn.Module): Loss function for kidney task.
        criterion_spleen (nn.Module): Loss function for spleen task.

    Returns:
        tuple: Tuple containing validation loss, accuracy for each task, and F1 score for each task.
    """
    device = "cuda" if cuda.is_available() else "cpu"
    # Create validation metrics variables
    val_loss, val_acc_bowel, val_acc_extra, val_acc_liver, val_acc_kidney, val_acc_spleen = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

    # Set model to evaluation mode
    model.eval()

    # Iterate over validation dataloader
    with inference_mode():
        for batch_idx, (X, y_bowel, y_extra, y_kidney, y_liver, y_spleen) in tqdm(enumerate(dataloader), desc="Validation", leave=False, total=len(dataloader)):
            # Move data to appropriate device
            X, y_bowel, y_extra, y_kidney, y_liver, y_spleen = X.to(device), y_bowel.to(device), y_extra.to(device), y_kidney.to(device), y_liver.to(device), y_spleen.to(device)

            # Forward pass
            bowel_out, extra_out, kidney_out, liver_out, spleen_out = model(X)

            # Calculate loss
            loss_bowel = criterion_bowel(bowel_out, y_bowel)
            loss_extra = criterion_extra(extra_out, y_extra)
            loss_kidney = criterion_kidney(kidney_out, y_kidney)
            loss_liver = criterion_liver(liver_out, y_liver)
            loss_spleen = criterion_spleen(spleen_out, y_spleen)

            # Accumulate loss
            val_loss += (loss_bowel.item() + loss_extra.item() + loss_liver.item() + loss_kidney.item() + loss_spleen.item())

            # Converting binary labels to true
            bowel_out=round(sigmoid(bowel_out))
            extra_out=round(sigmoid(extra_out))
            kidney_out=softmax(kidney_out,dim=1).argmax(dim=1)
            liver_out=softmax(liver_out,dim=1).argmax(dim=1)
            spleen_out=softmax(spleen_out,dim=1).argmax(dim=1)
            # Getting true labels for muticlass classification
            y_kidney=y_kidney.argmax(dim=1)
            y_liver=y_liver.argmax(dim=1)
            y_spleen=y_spleen.argmax(dim=1)

            # Compute accuracy for each task
            val_acc_bowel += binary_acc(bowel_out, y_bowel).item()
            val_acc_extra += binary_acc(extra_out, y_extra).item()
            val_acc_kidney += multi_acc(kidney_out, y_kidney).item()
            val_acc_liver += multi_acc(liver_out, y_liver).item()
            val_acc_spleen += multi_acc(spleen_out, y_spleen).item()

    # Compute average loss, accuracy, and F1 score
    num_batches = len(dataloader)
    val_loss /= (num_batches)
    val_acc_bowel /= num_batches
    val_acc_extra /= num_batches
    val_acc_kidney /= num_batches
    val_acc_liver /= num_batches
    val_acc_spleen /= num_batches

    return val_loss, val_acc_bowel, val_acc_extra, val_acc_kidney, val_acc_liver, val_acc_spleen

from tabulate import tabulate
# Define a function to print the DataFrame after each epoch
def print_model_train_results(results):
    df_results = pd.DataFrame(results)
    # Set 'Type' as the index
    df_results.set_index("Results", inplace=True)
    print(tabulate(df_results, headers='keys', tablefmt='fancy_grid'),"\n")

def train(model: nn.Module,
          train_dataloader: DataLoader,
          val_dataloader: DataLoader,
          optimizer: optim.Optimizer,
          criterion_bowel: nn.Module,
          criterion_extra: nn.Module,
          criterion_kidney: nn.Module,
          criterion_liver: nn.Module,
          criterion_spleen: nn.Module,
          wandb_init_params: Dict = {"project": "Demo_Project",
                                     "experiment": "Demo_Experiment",
                                     "hyperparameters": {
                                         "learning_rate": 0.0001,
                                         "epochs": 5,
                                         "batch_size": 32
                                     }},
          epochs: int = 5,
          lr_scheduler: optim.lr_scheduler = None,
          early_stopping: Dict = {"patience": 5},
          CUTMIX_ALPHA=None) -> Dict:
    """
    Train the model for a specified number of epochs.

    Parameters:
        model (nn.Module): The neural network model.
        train_dataloader (DataLoader): DataLoader for the training dataset.
        val_dataloader (DataLoader): DataLoader for the validation dataset.
        optimizer (optim.Optimizer): Optimizer for gradient descent.
        criterion_bowel (nn.Module): Loss function for the bowel task.
        criterion_extra (nn.Module): Loss function for the extra task.
        criterion_kidney (nn.Module): Loss function for the kidney task.
        criterion_liver (nn.Module): Loss function for the liver task.
        criterion_spleen (nn.Module): Loss function for the spleen task.
        accuracy_fn (nn.Module): Accuracy metric.
        wandb_init_params (Dict): Parameters for initializing Wandb.
        epochs (int): Number of training epochs. Default is 5.
        lr_scheduler (optim.lr_scheduler): Learning rate scheduler. Default is None.
        early_stopping (Dict): Dictionary containing early stopping parameters. Default is {"patience": 5}.

    Returns:
        Dict: A dictionary containing training and validation metrics.
    """

    device = "cuda" if cuda.is_available() else "cpu"
    model.to(device)
    images,temp_bowel, temp_extra, temp_kidney, temp_liver, temp_spleen = next(iter(val_dataloader))

    # Initialize Wandb
    wandb.init(project=wandb_init_params["project"], name=wandb_init_params["experiment"],
               config={"hyperparameters":wandb_init_params["hyperparameters"],
                       "criterions":{
                           "bowel":criterion_bowel,
                           "extravasation":criterion_extra,
                           "kidney":criterion_kidney,
                           "liver":criterion_liver,
                           "spleen":criterion_spleen
                       },
                       "metrics":{
                           "bowel,extravasation":{
                               "Accuracy":BinaryAccuracy
                           },
                           "kideny,liver,spleen":{
                               "num_classes": "3",
                               "Accuracy":MulticlassAccuracy
                           }
                       }})
    wandb.watch(model)

    # Getting accuracy and f1score
    binary_acc = BinaryAccuracy().to(device)
    multi_acc = MulticlassAccuracy(num_classes=3).to(device)

    best_val_loss = float('inf')
    best_epoch = 0
    no_improvement = 0

    results = defaultdict(list)
    results["model_name"]=wandb_init_params["experiment"]
    if CUTMIX_ALPHA is not None:
      print("Training With CutMix Augmentation")
    for epoch in tqdm(range(1, epochs + 1),desc="Epoch"):
        # Training
        if CUTMIX_ALPHA is None:
          train_loss, train_acc_bowel, train_acc_extra, train_acc_kidney, train_acc_liver,train_acc_spleen= train_step(
              model, train_dataloader,
              criterion_bowel, criterion_extra,
              criterion_kidney, criterion_liver,
              criterion_spleen,
              binary_acc,
              multi_acc,
              optimizer)
        else:
          train_loss, train_acc_bowel, train_acc_extra, train_acc_kidney, train_acc_liver,train_acc_spleen= train_step_cutmix(
               model, train_dataloader,
               criterion_bowel, criterion_extra,
               criterion_kidney, criterion_liver,
               criterion_spleen,
               binary_acc,
               multi_acc,
               optimizer,
               CUTMIX_ALPHA=CUTMIX_ALPHA)
        # Validation
        val_loss, val_acc_bowel, val_acc_extra, val_acc_kidney, val_acc_liver, val_acc_spleen = validation_step(
            model, val_dataloader,
            criterion_bowel, criterion_extra,
            criterion_kidney, criterion_liver,
            criterion_spleen,
            binary_acc,
            multi_acc)

        if lr_scheduler is not None:
            lr_scheduler.step(val_loss)
            print(f"lr:{optimizer.param_groups[0]['lr']}")

        wandb.log({
            "train_loss": train_loss,
            "train_acc_bowel": train_acc_bowel,
            "train_acc_extra": train_acc_extra,
            "train_acc_kidney": train_acc_kidney,
            "train_acc_liver": train_acc_liver,
            "train_acc_spleen": train_acc_spleen,
            "val_loss": val_loss,
            "val_acc_bowel": val_acc_bowel,
            "val_acc_extra": val_acc_extra,
            "val_acc_kidney": val_acc_kidney,
            "val_acc_liver": val_acc_liver,
            "val_acc_spleen": val_acc_spleen,
        })

        if lr_scheduler is not None:
            wandb.log({"learning_rate":optimizer.param_groups[0]['lr']})

        results["train_loss"].append(train_loss)
        results["train_acc_bowel"].append(train_acc_bowel)
        results["train_acc_extra"].append(train_acc_extra)
        results["train_acc_kidney"].append(train_acc_kidney)
        results["train_acc_liver"].append(train_acc_liver)
        results["train_acc_spleen"].append(train_acc_spleen)
        results["val_loss"].append(val_loss)
        results["val_acc_bowel"].append(val_acc_bowel)
        results["val_acc_extra"].append(val_acc_extra)
        results["val_acc_kidney"].append(val_acc_kidney)
        results["val_acc_liver"].append(val_acc_liver)
        results["val_acc_spleen"].append(val_acc_spleen)

        if lr_scheduler is not None:
            results["learning_rate"].append(optimizer.param_groups[0]['lr'])

        print(f"Epoch {epoch}:")
        print_model_train_results({
            "Results":["Train","Validation"],
            "loss":[train_loss,val_loss],
            "bowel": [train_acc_bowel,val_acc_bowel],
            "extravation": [train_acc_extra,val_acc_extra],
            "kidney": [train_acc_kidney,val_acc_kidney],
            "liver": [train_acc_liver,val_acc_liver],
            "spleen": [train_acc_spleen,val_acc_spleen]
            })

        # Check for early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improvement = 0
            print(f"Saving new best model")
            # Delete previous best epoch model if it exists
            if best_epoch > 0:
              previous_model_path = f"{wandb_init_params['experiment']}/epoch_{best_epoch - 1}.pth"
              if os.path.exists(previous_model_path):
                os.remove(previous_model_path)
            utils.save_model(model, f"{wandb_init_params['experiment']}/epoch_{epoch}", f"{wandb_init_params['experiment']}.pth")
        else:
            no_improvement += 1

        if no_improvement >= early_stopping["patience"]:
            wandb.log({
                "train_loss": train_loss,
                "train_acc_bowel": train_acc_bowel,
                "train_acc_extra": train_acc_extra,
                "train_acc_kidney": train_acc_kidney,
                "train_acc_liver": train_acc_liver,
                "train_acc_spleen": train_acc_spleen,
                "val_loss": val_loss,
                "val_acc_bowel": val_acc_bowel,
                "val_acc_extra": val_acc_extra,
                "val_acc_kidney": val_acc_kidney,
                "val_acc_liver": val_acc_liver,
                "val_acc_spleen": val_acc_spleen,
            })
            if lr_scheduler is not None:
                wandb.log({"learning_rate":optimizer.param_groups[0]['lr']})

            results["train_loss"].append(train_loss)
            results["train_acc_bowel"].append(train_acc_bowel)
            results["train_acc_extra"].append(train_acc_extra)
            results["train_acc_kidney"].append(train_acc_kidney)
            results["train_acc_liver"].append(train_acc_liver)
            results["train_acc_spleen"].append(train_acc_spleen)
            results["val_loss"].append(val_loss)
            results["val_acc_bowel"].append(val_acc_bowel)
            results["val_acc_extra"].append(val_acc_extra)
            results["val_acc_kidney"].append(val_acc_kidney)
            results["val_acc_liver"].append(val_acc_liver)
            results["val_acc_spleen"].append(val_acc_spleen)

            if lr_scheduler is not None:
                results["learning_rate"].append(optimizer.param_groups[0]['lr'])

            wandb.log({"early_stopping_epoch": epoch})
            wandb.log({"early_stopping_reason": "No improvement in validation loss"})
            print(f"Early stopping at epoch {epoch}")
            print(f"Saving last epoch model")
            utils.save_model(model, f"{wandb_init_params['experiment']}/epoch_{epoch}", f"{wandb_init_params['experiment']}.pth")
            break

    print(f"Saving last epoch model")
    utils.save_model(model, f"{wandb_init_params['experiment']}/epoch_{epoch}", f"{wandb_init_params['experiment']}.pth")
    print(f"Best validation loss: {best_val_loss} at epoch {best_epoch}")
    model.eval()
    # Save the model in the exchangeable ONNX format
    onnx.export(model, images.to(device), f"{wandb_init_params['experiment']}.onnx")
    wandb.save(f"{wandb_init_params['experiment']}.onnx")
    wandb.finish()
    return results

Writing torch_helper_functions/multilabel/train_engine.py


In [6]:
%%writefile torch_helper_functions/plotting_utils.py
"""
This contains utility realted to plotting model information.

## Functions:
  `plot_loss_accuracy_curve`:plot model training and evaluation loss and accuracy curves
"""
import matplotlib.pyplot as plt
from typing import Dict,List
import torch
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix
import torch
import torch.nn.functional as F
import seaborn as sns

def generate_confusion_matrix(model, dataloader):
    model.eval()
    all_preds = {task: [] for task in ['Bowel', 'Extra', 'Kidney', 'Liver', 'Spleen']}
    all_labels = {task: [] for task in ['Bowel', 'Extra', 'Kidney', 'Liver', 'Spleen']}
    device = "cuda" if torch.cuda.is_available() else "cpu"

    with torch.no_grad():
        for images, y_bowel, y_extra, y_kidney, y_liver, y_spleen in dataloader:
            images = images.to(device)

            # Forward pass
            bowel_out, extra_out, kidney_out, liver_out, spleen_out = model(images)

            # Convert logits to predictions
            bowel_preds = (torch.sigmoid(bowel_out) > 0.5).int().cpu().numpy().flatten()
            extra_preds = (torch.sigmoid(extra_out) > 0.5).int().cpu().numpy().flatten()
            kidney_preds = np.argmax(F.softmax(kidney_out, dim=1).cpu().numpy(), axis=1).flatten()
            liver_preds = np.argmax(F.softmax(liver_out, dim=1).cpu().numpy(), axis=1).flatten()
            spleen_preds = np.argmax(F.softmax(spleen_out, dim=1).cpu().numpy(), axis=1).flatten()

            # Append predictions to the dictionary
            all_preds['Bowel'].append(bowel_preds)
            all_preds['Extra'].append(extra_preds)
            all_preds['Kidney'].append(kidney_preds)
            all_preds['Liver'].append(liver_preds)
            all_preds['Spleen'].append(spleen_preds)

            # Append true labels to the dictionary (already on CPU)
            all_labels['Bowel'].append(y_bowel.numpy())
            all_labels['Extra'].append(y_extra.numpy())
            all_labels['Kidney'].append(torch.argmax(y_kidney, dim=1).numpy())
            all_labels['Liver'].append(torch.argmax(y_liver, dim=1).numpy())
            all_labels['Spleen'].append(torch.argmax(y_spleen, dim=1).numpy())

    # Concatenate the lists within each task
    for task in ['Bowel', 'Extra', 'Kidney', 'Liver', 'Spleen']:
        all_preds[task] = np.concatenate(all_preds[task])
        all_labels[task] = np.concatenate(all_labels[task])

    # Create 2x2 subplot for confusion matrices
    fig, axes = plt.subplots(3, 2, figsize=(12, 12))

    # Flatten axes for easier indexing
    axes = axes.flatten()

    # Create confusion matrix for each task
    for i, task in enumerate(['Bowel', 'Extra', 'Kidney', 'Liver', 'Spleen']):
        cm = confusion_matrix(all_labels[task], all_preds[task])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
        axes[i].set_title(f'Confusion Matrix for {task}')
        axes[i].set_xlabel('Predicted')
        axes[i].set_ylabel('Actual')

    # Hide the remaining subplot
    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

def plot_model_history_curves(model_history: Dict):
    """
    Plots the training history including loss and metrics for each class.

    Parameters:
    - model_history (dict): A dictionary containing training history data.
                      It should have keys like 'train_loss', 'val_loss',
                      'train_acc_classname', 'val_acc_classname', etc.

    Returns:
    - None
    """
    epochs = range(1, len(model_history["train_loss"]) + 1)
    # Plot train and validation losses for each class
    fig, axs = plt.subplots(2, 3, figsize=(15, 10))
    title_position = (0.5, 1)  # Adjust the position as needed
    fig.suptitle(f"{model_history.get('model_name', 'Model')} Training History", fontsize=18,
                 position=title_position, fontweight='bold')
    axs = axs.ravel()
    handles = []
    labels = []
    train_line, = axs[0].plot(epochs, model_history['train_loss'], 'r', label='Train Loss')
    val_line, = axs[0].plot(epochs, model_history['val_loss'], 'b', label='Validation Loss')
    axs[0].set_title('Train and Validation Loss - All Classes')
    axs[0].set_xlabel('Epochs')
    axs[0].set_ylabel('Loss')
    axs[0].grid(True)
    handles.extend([train_line, val_line])
    labels.extend([train_line.get_label(), val_line.get_label()])
    classes = ['bowel', 'extra', 'kidney','liver', 'spleen']

    colors = ['violet', 'orange']
    metrics = ['acc']

    for i, class_name in enumerate(classes):
        train_line, = axs[i + 1].plot(epochs, model_history[f'train_acc_{class_name}'], color=colors[0], linestyle='-', label=f'Train Accuracy')
        val_line, = axs[i + 1].plot(epochs, model_history[f'val_acc_{class_name}'], color=colors[1], linestyle='--', label=f'Validation Accuracy')
        if i == 0:
          handles.extend([train_line, val_line])
          labels.extend([train_line.get_label(), val_line.get_label()])

        axs[i + 1].set_title(f'Train and Validation Metrics - {class_name.capitalize()}')
        axs[i + 1].set_xlabel('Epochs')
        axs[i + 1].set_ylabel('Metrics')
        axs[i + 1].grid(True)

    # Create a single legend for all subplots
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), shadow=True, ncol=3)

    plt.tight_layout()
    plt.show()

# def plot_images_with_predictions(test_dataloader:torch.utils.data.DataLoader,
#                                  model:torch.nn.Module,
#                                  label_names:List[str]=None):
#     """
#     #### Plots images from the test dataloader along with predicted labels and probabilities.

#     ## Parameters:
#       `test_dataloader (torch.utils.data.DataLoader)`: Dataloader for the test dataset.
#       `model (torch.nn.Module)`: The trained model for making predictions.
#       `label_names (list)`: List of label names. Default is None.

#     ## Returns:
#       None

#     ## Example:
#       plot_images_with_predictions(test_dataloader, model, label_names=['bowel_injury', 'extravasation_injury', ...])
#     """
#     # Setting device
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     # Set model to evaluation mode
#     model.eval()
#     # Get one batch of test data
#     images, labels = next(iter(test_dataloader))
#     images = images.to(device)

#     # Forward pass
#     with torch.inference_mode():
#         outputs = model(images)
#         probabilities = torch.sigmoid(outputs)

#     # Convert probabilities and labels to numpy arrays
#     probabilities = probabilities.cpu().numpy()
#     labels = labels.numpy()

#     # Plot images with predicted labels and probabilities
#     fig, axes = plt.subplots(2, 5, figsize=(15, 8))
#     fig.suptitle('Images with Predicted Labels and Probabilities', fontsize=16)
#     axes = axes.flatten()

#     for i in range(10):
#         image = images[i].cpu().numpy().transpose((1, 2, 0))
#         image = np.clip(image, 0, 1)  # Clip image values to [0, 1] range
#         axes[i].imshow(image)
#         axes[i].axis('off')

#         # Get labels and probabilities
#         label_indices = np.arange(len(probabilities[i]))
#         probs = probabilities[i]

#         # Sort labels based on probabilities
#         sorted_indices = np.argsort(-probs)
#         sorted_probs = probs[sorted_indices]
#         sorted_labels = label_indices[sorted_indices]

#         # Get label names if provided
#         if label_names:
#             label_info = [f'{label_names[j]}: {sorted_probs[j]:.2f}' for j in sorted_labels]
#         else:
#             label_info = [f'Label {sorted_labels[j]}: {sorted_probs[j]:.2f}' for j in range(len(sorted_probs))]

#         # Set title with labels and probabilities
#         title = '\n'.join(label_info)
#         axes[i].set_title(title)

#     plt.tight_layout()
#     plt.show()

# def plot_roc_curves(test_dataloader, model, label_names=None):
#     """
#     #### Plots ROC curves for each label in the multi-label classification task.

#     ## Parameters:
#       `test_dataloader (torch.utils.data.DataLoader)`: Dataloader for the test dataset.
#       `model (torch.nn.Module)`: The trained model for evaluation.
#       `label_names (list)`: List of label names. Default is None.

#     ## Returns:
#        None

#     ## Example:
#         plot_roc_curves(test_dataloader, model, label_names=['Label 0', 'Label 1', ...])
#     """
#     # Setting device
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     # Set model to evaluation mode
#     model.eval()

#     true_labels = []
#     predicted_probabilities = []

#     with torch.inference_mode():
#         for images, labels in test_dataloader:
#             images = images.to(device)

#             outputs = model(images)
#             probabilities = torch.sigmoid(outputs).cpu().numpy()

#             true_labels.extend(labels.numpy())
#             predicted_probabilities.extend(probabilities)

#     # Convert true labels and predicted probabilities to numpy arrays
#     true_labels = np.array(true_labels)
#     predicted_probabilities = np.array(predicted_probabilities)

#     # Get label names if provided
#     target_names = label_names if label_names else [f'Label {i}' for i in range(true_labels.shape[1])]

#     # Compute and plot ROC curve for each label
#     plt.figure(figsize=(10, 7))
#     for label_idx in range(true_labels.shape[1]):
#         fpr, tpr, _ = roc_curve(true_labels[:, label_idx], predicted_probabilities[:, label_idx])
#         roc_auc = auc(fpr, tpr)
#         plt.plot(fpr, tpr, label=f'{target_names[label_idx]} (AUC = {roc_auc:.2f})')

#     plt.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random Guess')
#     plt.xlabel('False Positive Rate')
#     plt.ylabel('True Positive Rate')
#     plt.title('ROC Curves for Multi-label Classification')
#     plt.legend()
#     plt.grid(True)
#     plt.show()

Writing torch_helper_functions/plotting_utils.py


In [7]:
%%writefile torch_helper_functions/multilabel/data_setup.py
"""
Contains functionality for creating PyTorch DataLoader's for
image classification data and related utilities.
"""
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from typing import List, Tuple, Dict
from pathlib import Path
from PIL import Image
import os
from sklearn.model_selection import train_test_split
import pandas as pd

def find_classes(label_columns: List[str]) -> Tuple[List[str], Dict[str, List[int]]]:
    """
    #### Identify classes from the given list of label columns and assign numerical labels.

    ## Parameters:
      `label_columns (List[str])`: List of class labels.

    ## Returns:
      `Tuple[List[str], Dict[str, List[int]]]`: A tuple containing a list of class names
          and a dictionary mapping class names to numerical labels.

    ## Exampel:
       label_columns = ["dog","cat"]
       classes,class_to_idx = find_classes(label_colums)
    """
    # 1. Get the class names by scanning the target directory
    classes = label_columns

    # 2. Raise an error if class names not found
    if not classes:
        raise FileNotFoundError(f"Couldn't find any classes in {label_columns}.")

    # 3. Create a dictionary of index labels (computers prefer numerical rather than string labels)
    class_to_idx = {}
    for index, class_name in enumerate(label_columns):
        temp = [0] * len(label_columns)
        temp[index] = 1
        class_to_idx[class_name] = temp
    return classes, class_to_idx

class Images_From_DataFrame(Dataset):
    """
    #### Dataset class for loading images and labels from a DataFrame.

    ## Parameters:
      `df (pd.DataFrame)`: DataFrame containing image paths and labels.
      `image_path_column (str)`: Name of the column containing image paths.
      `label_columns (List[str])`: List of column names containing labels.
      `transform (Optional[transforms.Compose])`: Optional transforms to be applied to images.

    ## Returns:
      `Tuple[torch.Tensor, torch.Tensor]`: A tuple containing the image tensor and its label tensor.

    ## Example:
      train_data=Images_From_DataFrame(df,'image_path',["dog","cat"],transform)
    """
    def __init__(self, df: pd.DataFrame, image_path_column: str, label_columns: List[str], transform:transforms.Compose=None,channels:int=1):
        self.paths = df[image_path_column].apply(Path).tolist()
        self.df = df
        self.label_columns = label_columns
        self.transform = transform
        self.classes, self.class_to_idx = find_classes(label_columns)
        self.channels=channels

    def load_image(self, index: int):
        """Load image from the given index."""
        image_path = self.paths[index]
        if self.channels==3:
          return Image.open(image_path).convert('RGB')
        return Image.open(image_path)

    def __len__(self) -> int:
        """Return the number of datapoints in the dataset."""
        return len(self.paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return the image and its label at the given index."""
        img = self.load_image(index)
        label = self.df[self.label_columns].iloc[index].values.tolist()
        if self.transform:
            label=torch.tensor(label, dtype=torch.float32)
            y_bowel, y_extra, y_kidney, y_liver, y_spleen=label[0:1],label[1:2],label[2:5],label[5:8],label[8:]
            return self.transform(img),y_bowel, y_extra, y_kidney, y_liver, y_spleen
        else:
            label=torch.tensor(label, dtype=torch.float32)
            y_bowel, y_extra, y_kidney, y_liver, y_spleen=label[0:1],label[1:2],label[2:5],label[5:8],label[8:]
            return img, y_bowel, y_extra, y_kidney, y_liver, y_spleen

    def __repr__(self):
        """Return a string representation of the dataset."""
        return (f"Dataset Images_From_DataFrame\n\
                  Number of datapoints: {len(self.paths)}\n\
                  StandardTransform\nTransform: {self.transform}")

def create_dataloaders_from_dataframe(
    df: pd.DataFrame,
    image_path_column: str,
    label_columns: List[str],
    train_transform: transforms.Compose,
    val_transform: transforms.Compose,
    test_transform: transforms.Compose,
    batch_size: int=32,
    validation_split: float = 0.2,
    test_split: float = 0.1,
    num_workers: int = os.cpu_count(),
    channels:int=1,
    collate_fn=None) -> Tuple[DataLoader, DataLoader, DataLoader, List[str]]:
    """
    #### Create DataLoader instances for training, validation, and testing from a DataFrame.

    ## Parameters
      `df (pd.DataFrame)`: DataFrame containing image paths and labels.
      `image_path_column (str)`: Name of the column containing image paths.
      `label_columns (List[str])`: List of column names containing labels.
      `transform (transforms.Compose)`: Transforms to be applied to images.
      `batch_size (int)`: Batch size for DataLoader.
      `validation_split (float)`: Percentage of data to use for validation. Default is 0.2.
      `test_split (float)`: Percentage of data to use for testing. Default is 0.1.
      `num_workers (int)`: Number of subprocesses to use for data loading. Default is CPU count.

    ## Returns:
        `Tuple[DataLoader, DataLoader, DataLoader, List[str]]`: A tuple containing
          DataLoaders for training, validation, and testing, and a list of class names.

    ## Example:
       train_dataloader, val_dataloader, test_dataloader, classes=create_dataloaders_from_dataframe(df,
                                                                                                    'image_path',
                                                                                                    ["dog","cat"],
                                                                                                    transform.32,
                                                                                                    0.2,
                                                                                                    0.1,
                                                                                                    0)
    """
    # Split the DataFrame into train and test data
    train_data, test_data = train_test_split(df, test_size=test_split,random_state=42)

    # Further split the train data into train and validation data
    train_data, val_data = train_test_split(train_data, test_size=validation_split, random_state=42)
    # Create datasets and dataloaders for train, validation, and test data
    train_dataset = Images_From_DataFrame(df=train_data,
                                           image_path_column=image_path_column,
                                           label_columns=label_columns,
                                           transform=train_transform,
                                           channels=channels)
    val_dataset = Images_From_DataFrame(df=val_data,
                                         image_path_column=image_path_column,
                                         label_columns=label_columns,
                                         transform=val_transform,
                                         channels=channels)
    test_dataset = Images_From_DataFrame(df=test_data,
                                          image_path_column=image_path_column,
                                          label_columns=label_columns,
                                          transform=test_transform,
                                          channels=channels)
    # Get classes
    classes = train_dataset.classes
    # Create DataLoaders
    train_dataloader = DataLoader(dataset=train_dataset,
                                   batch_size=batch_size,
                                   num_workers=num_workers,
                                   shuffle=True,
                                   collate_fn=collate_fn)  # Shuffle training data
    val_dataloader = DataLoader(dataset=val_dataset,
                                 batch_size=batch_size,
                                 num_workers=num_workers,
                                 shuffle=False)  # No need to shuffle validation data
    test_dataloader = DataLoader(dataset=test_dataset,
                                  batch_size=batch_size,
                                  num_workers=num_workers,
                                  shuffle=False)  # No need to shuffle test data
    print(f"Each 1 instance in dataloader={batch_size} data points.")
    print(f"Train DataLoader contains: {len(train_dataloader)} instance = {train_data.shape[0]} data points.")
    print(f"Validation DataLoader contains: {len(val_dataloader)} instance = {val_data.shape[0]} data points.")
    print(f"Test DataLoader contains: {len(test_dataloader)} instance = {test_data.shape[0]} data points.")
    return train_dataloader, val_dataloader, test_dataloader, classes

Writing torch_helper_functions/multilabel/data_setup.py
